In [4]:
!pip install scikit-learn scipy pandas numpy tqdm joblib faiss-cpu torch -q
import warnings
warnings.filterwarnings('ignore')
print('✅ Cài đặt thư viện hoàn tất')

✅ Cài đặt thư viện hoàn tất


In [5]:
import os, glob, gc, time, math
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm
from collections import defaultdict
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import faiss

# ─────────────────────────────────────────────
CONFIG = {
    'data_dir'        : '/kaggle/input/datasets/js042710/second3t1k/data DM/data DM',
    'output_dir'      : '/kaggle/working/lightgcn_model',
    'checkpoint_dir'  : '/kaggle/working/checkpoints',
    # ── LightGCN ──
    'emb_dim'         : 128,     # tăng từ 64
    'n_layers'        : 3,
    'lr'              : 1e-3,
    'decay'           : 1e-3,    # tăng từ 1e-4 → kiểm soát reg loss
    'epochs'          : 20,
    'batch_size'      : 8192,
    'grad_clip'       : 1.0,     # gradient clipping max norm
    'device'          : 'cuda' if torch.cuda.is_available() else 'cpu',
    # ── Data ──
    'min_interactions': 20,
    'chunk_size'      : 500_000,
    'recency_halflife': 60,      # half-life recency decay (ngày)
    # ── Negative sampling ──
    'neg_pool_size'   : 2_000_000,  # precompute pool để tránh O(n) per sample
}
# ─────────────────────────────────────────────

os.makedirs(CONFIG['output_dir'],    exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

print(f"Device      : {CONFIG['device']}")
print(f"emb_dim     : {CONFIG['emb_dim']}   |  n_layers : {CONFIG['n_layers']}")
print(f"decay       : {CONFIG['decay']}  |  grad_clip: {CONFIG['grad_clip']}")
print(f"halflife    : {CONFIG['recency_halflife']} days")
print(f"neg_pool    : {CONFIG['neg_pool_size']:,} samples")

Device      : cuda
emb_dim     : 128   |  n_layers : 3
decay       : 0.001  |  grad_clip: 1.0
halflife    : 60 days
neg_pool    : 2,000,000 samples


## 1. Data Loader với Timestamp Chuẩn

**Sửa so với v1:**
- `_raw[(user_str, msid)] = [count, last_ts_unix]` — lưu số lần nghe và timestamp cuối cùng
- `global_max_ts` cập nhật toàn cục qua tất cả chunks (không phải per-chunk)
- **Bug fix**: lọc NaT *trước* `.astype('int64')` — NaT convert sang `int64` cho ra số âm cực lớn, không phải NaN
- `build_matrix()`: weight = `count × exp(−days_since_last_play / halflife)` dùng global max
- Sau filter, rebuild `user_item_ts_matrix[new_uid][new_iid] = last_ts` cho temporal split

In [6]:
class InteractionAccumulator:
    """
    Tích lũy tương tác từ nhiều file CSV/XLSX.
    Lưu (count, last_ts_unix) per (user, item) — tính recency weight đúng với global max.
    """
    def __init__(self):
        self.user2idx   = {}
        self.item2idx   = {}
        self.idx2user   = []
        self.idx2item   = []
        self.item_meta  = {}
        # (uid_str, msid) → [play_count, last_ts_unix_seconds]
        self._raw            = defaultdict(lambda: [0, 0.0])
        self.global_max_ts   = 0.0   # ← FIX: cập nhật sau mỗi chunk, không reset
        self.user_raw_items  = defaultdict(set)
        self.user_item_ts_matrix = None   # set sau build_matrix
        self.n_rows_processed    = 0
        self._n_ts_missing        = 0     # đếm hàng không có timestamp hợp lệ

    def _get_or_create(self, mapping, reverse, key):
        if key not in mapping:
            mapping[key] = len(reverse)
            reverse.append(key)
        return mapping[key]

    def add_dataframe(self, df: pd.DataFrame):
        required = ['user_id', 'recording_msid', 'track_name', 'artist_name']
        has_ts   = 'timestamp' in df.columns
        df = df[required + (['timestamp'] if has_ts else [])].dropna(subset=required).copy()

        # ── Metadata bài hát ──────────────────────────────────────────────────
        meta = df[['recording_msid','track_name','artist_name']].drop_duplicates('recording_msid')
        for r in meta.itertuples(index=False):
            if r.recording_msid not in self.item_meta:
                self.item_meta[r.recording_msid] = {
                    'track_name' : str(r.track_name).strip(),
                    'artist_name': str(r.artist_name).strip(),
                }

        df['user_id'] = df['user_id'].astype(str)

        # ── Timestamp ─────────────────────────────────────────────────────────
        if has_ts:
            df['ts_parsed'] = pd.to_datetime(df['timestamp'], errors='coerce')
            n_before = len(df)
            df = df[df['ts_parsed'].notna()].copy()   # FIX: lọc NaT TRƯỚC convert
            self._n_ts_missing += n_before - len(df)

            # Sau khi lọc, convert an toàn — không còn NaT nào
            df['ts_unix'] = df['ts_parsed'].astype('int64') / 1e9   # ns → giây
            chunk_max = df['ts_unix'].max()
            if chunk_max > self.global_max_ts:
                self.global_max_ts = chunk_max          # ← cập nhật global max
            agg_cols = {'count': ('recording_msid','count'), 'last_ts': ('ts_unix','max')}
        else:
            df['ts_unix'] = 0.0
            agg_cols = {'count': ('recording_msid','count'), 'last_ts': ('ts_unix','max')}

        # ── Vectorized groupby — nhanh hơn itertuples từng dòng ──────────────
        agg = df.groupby(['user_id','recording_msid'], sort=False).agg(**agg_cols).reset_index()

        for r in agg.itertuples(index=False):
            uid_str, msid = r.user_id, r.recording_msid
            self._get_or_create(self.user2idx, self.idx2user, uid_str)
            self._get_or_create(self.item2idx, self.idx2item, msid)
            entry = self._raw[(uid_str, msid)]
            entry[0] += int(r.count)
            if r.last_ts > entry[1]:
                entry[1] = float(r.last_ts)
            self.user_raw_items[uid_str].add(msid)

        self.n_rows_processed += len(df)

    def build_matrix(self, min_interactions: int = 20):
        hl = CONFIG['recency_halflife']
        t0 = time.time()
        n_pairs = len(self._raw)
        print(f'[build_matrix] {n_pairs:,} unique (user, item) pairs')
        if self.global_max_ts > 0:
            print(f'  Global max timestamp : {pd.Timestamp(self.global_max_ts, unit="s")}')
        else:
            print('  ⚠ Không có dữ liệu timestamp — recency weight = 1.0 cho tất cả')
        if self._n_ts_missing:
            print(f'  ⚠ Đã bỏ {self._n_ts_missing:,} hàng có timestamp không hợp lệ (NaT)')

        rows, cols, data = [], [], []
        weights_for_log  = []
        for (uid_str, msid), (count, last_ts) in self._raw.items():
            if uid_str not in self.user2idx or msid not in self.item2idx:
                continue
            uid = self.user2idx[uid_str]
            iid = self.item2idx[msid]
            if self.global_max_ts > 0 and last_ts > 0:
                days_ago = (self.global_max_ts - last_ts) / 86400.0
                recency  = math.exp(-days_ago / hl)
            else:
                recency  = 1.0
            w = float(count) * recency
            rows.append(uid); cols.append(iid); data.append(w)
            weights_for_log.append(w)

        # ── Log thống kê trọng số ─────────────────────────────────────────────
        if weights_for_log:
            w_arr = np.array(weights_for_log)
            print(f'  Weight stats → min={w_arr.min():.4f} | max={w_arr.max():.2f} '
                  f'| mean={w_arr.mean():.4f} | median={np.median(w_arr):.4f}')
            pct_recent = np.mean(w_arr > 0.5) * 100
            print(f'  {pct_recent:.1f}% tương tác có weight > 0.5 (nghe ≤ {hl*0.693:.0f} ngày gần đây)')

        mat = sp.csr_matrix(
            (data, (rows, cols)),
            shape=(len(self.idx2user), len(self.idx2item)),
            dtype=np.float32
        )
        print(f'  Ma trận ban đầu  : {mat.shape[0]:,} × {mat.shape[1]:,} '
              f'| {mat.nnz:,} interactions | sparsity={1-mat.nnz/(mat.shape[0]*mat.shape[1]+1e-9):.5f}')

        # ── Lọc theo min_interactions ─────────────────────────────────────────
        if min_interactions > 1:
            mat_bin = (mat > 0).astype(np.float32)
            ic_cnt  = np.asarray(mat_bin.sum(axis=0)).flatten()
            uc_cnt  = np.asarray(mat_bin.sum(axis=1)).flatten()
            vi = ic_cnt >= min_interactions
            vu = uc_cnt >= min_interactions
            n_user_drop = (~vu).sum()
            n_item_drop = (~vi).sum()
            mat           = mat[vu][:, vi]
            self.idx2user = [u for u, v in zip(self.idx2user, vu) if v]
            self.idx2item = [i for i, v in zip(self.idx2item, vi) if v]
            self.user2idx = {u: i for i, u in enumerate(self.idx2user)}
            self.item2idx = {i: j for j, i in enumerate(self.idx2item)}
            print(f'  Sau lọc (≥{min_interactions} interactions): '
                  f'{mat.shape[0]:,} users (-{n_user_drop:,}), '
                  f'{mat.shape[1]:,} items (-{n_item_drop:,})')
            print(f'  Ma trận cuối     : {mat.nnz:,} interactions | '
                  f'avg {mat.nnz/mat.shape[0]:.1f} items/user')

        # ── Rebuild timestamp lookup với new indices ───────────────────────────
        self.user_item_ts_matrix = defaultdict(dict)
        n_ts_records = 0
        for (uid_str, msid), (_, last_ts) in self._raw.items():
            if uid_str in self.user2idx and msid in self.item2idx:
                self.user_item_ts_matrix[self.user2idx[uid_str]][self.item2idx[msid]] = last_ts
                if last_ts > 0:
                    n_ts_records += 1
        pct_with_ts = n_ts_records / (mat.nnz + 1e-9) * 100
        print(f'  Timestamp records: {n_ts_records:,} ({pct_with_ts:.1f}% interactions có timestamp)')
        print(f'[build_matrix] Hoàn tất trong {time.time()-t0:.1f}s')
        return mat


# ── Đọc toàn bộ file ──────────────────────────────────────────────────────────
accumulator = InteractionAccumulator()
all_files   = sorted(glob.glob(os.path.join(CONFIG['data_dir'], '**', '*.csv'),  recursive=True))
all_files  += sorted(glob.glob(os.path.join(CONFIG['data_dir'], '**', '*.xlsx'), recursive=True))
print(f'Tìm thấy {len(all_files)} file')

USECOLS_CSV  = ['user_id', 'recording_msid', 'track_name', 'artist_name', 'timestamp']
USECOLS_XLSX = ['user_id', 'recording_msid', 'track_name', 'artist_name']

t0 = time.time()
n_err = 0
for fpath in tqdm(all_files, desc='Loading'):
    try:
        if fpath.endswith('.xlsx'):
            accumulator.add_dataframe(pd.read_excel(fpath, usecols=USECOLS_XLSX))
        else:
            for chunk in pd.read_csv(fpath, usecols=USECOLS_CSV,
                                     chunksize=CONFIG['chunk_size'], low_memory=False):
                accumulator.add_dataframe(chunk)
                del chunk; gc.collect()
    except Exception as e:
        print(f'  ⚠ Lỗi [{os.path.basename(fpath)}]: {e}')
        n_err += 1

elapsed = time.time() - t0
print(f'\nĐọc xong: {accumulator.n_rows_processed:,} rows | {elapsed:.0f}s | {n_err} lỗi')
print(f'Users raw: {len(accumulator.idx2user):,} | Items raw: {len(accumulator.idx2item):,}')

user_item_matrix = accumulator.build_matrix(min_interactions=CONFIG['min_interactions'])

Tìm thấy 32 file


Loading: 100%|██████████| 32/32 [22:15<00:00, 41.74s/it]



Đọc xong: 85,649,356 rows | 1336s | 0 lỗi
Users raw: 25,700 | Items raw: 11,261,940
[build_matrix] 25,780,276 unique (user, item) pairs
  Global max timestamp : 2026-03-10 00:06:56
  Weight stats → min=0.0000 | max=10460.19 | mean=0.9013 | median=0.0159
  43.9% tương tác có weight > 0.5 (nghe ≤ 42 ngày gần đây)
  Ma trận ban đầu  : 25,700 × 11,261,940 | 25,780,276 interactions | sparsity=0.99991
  Sau lọc (≥20 interactions): 21,765 users (-3,935), 133,692 items (-11,128,248)
  Ma trận cuối     : 6,374,876 interactions | avg 292.9 items/user
  Timestamp records: 6,353,371 (99.7% interactions có timestamp)
[build_matrix] Hoàn tất trong 60.0s


## 2. Temporal Train/Test Split

**Logic:** Với mỗi user, sắp xếp items theo `last_ts_unix` tăng dần.  
20% items có timestamp *mới nhất* → **test set** | Còn lại → **train set**

Nếu user không có timestamp (ts=0), items vẫn được split nhưng thứ tự là tùy ý (vẫn đúng về mặt kỹ thuật, nhưng không phải "temporal" thực sự — sẽ được cảnh báo trong log).

In [7]:
def split_train_test_temporal(user_item_matrix, accumulator,
                              test_ratio=0.2, min_items=5):
    """
    Temporal split dùng trực tiếp CSR indptr — không cần convert sang LIL.
    Với mỗi user: sort items theo last_ts, 20% cuối → test.
    """
    t0         = time.time()
    n_users    = user_item_matrix.shape[0]
    indptr     = user_item_matrix.indptr
    idx_arr    = user_item_matrix.indices
    data_arr   = user_item_matrix.data

    tr_r, tr_c, tr_d = [], [], []
    te_r, te_c, te_d = [], [], []
    n_no_ts = 0       # users không có timestamp data
    n_skipped = 0     # users < min_items

    for uid in range(n_users):
        s, e  = indptr[uid], indptr[uid + 1]
        items = idx_arr[s:e]
        vals  = data_arr[s:e]

        if len(items) < min_items:
            for iid, v in zip(items, vals):
                tr_r.append(uid); tr_c.append(int(iid)); tr_d.append(float(v))
            n_skipped += 1
            continue

        ts_dict = accumulator.user_item_ts_matrix.get(uid, {})
        ts_vals = [ts_dict.get(int(iid), 0.0) for iid in items]

        if max(ts_vals) == 0.0:
            n_no_ts += 1   # cảnh báo nhưng vẫn xử lý

        # sort theo timestamp tăng dần; gần nhất → cuối mảng → vào test
        order   = sorted(range(len(items)), key=lambda k: ts_vals[k])
        n_test  = max(1, int(len(items) * test_ratio))

        for rank, k in enumerate(order):
            iid, v = int(items[k]), float(vals[k])
            if rank < len(order) - n_test:
                tr_r.append(uid); tr_c.append(iid); tr_d.append(v)
            else:
                te_r.append(uid); te_c.append(iid); te_d.append(v)

    shape    = user_item_matrix.shape
    train_m  = sp.csr_matrix((tr_d, (tr_r, tr_c)), shape=shape, dtype=np.float32)
    test_m   = sp.csr_matrix((te_d, (te_r, te_c)), shape=shape, dtype=np.float32)

    print(f'[temporal_split] Hoàn tất trong {time.time()-t0:.1f}s')
    print(f'  Train : {train_m.nnz:,} interactions  ({train_m.nnz/(train_m.nnz+test_m.nnz)*100:.1f}%)')
    print(f'  Test  : {test_m.nnz:,}  interactions  ({test_m.nnz/(train_m.nnz+test_m.nnz)*100:.1f}%)')
    print(f'  Users giữ nguyên train (< {min_items} items): {n_skipped:,}')
    if n_no_ts:
        print(f'  ⚠ {n_no_ts:,} users không có timestamp — test split theo vị trí (không theo thời gian)')
    return train_m, test_m


print('Đang tạo Temporal Train/Test Split...')
train_matrix, test_matrix = split_train_test_temporal(
    user_item_matrix, accumulator, test_ratio=0.2, min_items=5
)

Đang tạo Temporal Train/Test Split...
[temporal_split] Hoàn tất trong 10.2s
  Train : 5,108,014 interactions  (80.1%)
  Test  : 1,266,862  interactions  (19.9%)
  Users giữ nguyên train (< 5 items): 2,901


## 3. Graph Construction & Kiến Trúc LightGCN

In [8]:
def create_lightgcn_graph(user_item_matrix):
    """Bipartite Graph Laplacian chuẩn hóa đối xứng D^{-1/2} A D^{-1/2}."""
    t0 = time.time()
    n_users, n_items = user_item_matrix.shape
    n_nodes = n_users + n_items

    rows, cols   = user_item_matrix.nonzero()
    cols_shifted = cols + n_users

    edge_i = np.concatenate([rows, cols_shifted])
    edge_j = np.concatenate([cols_shifted, rows])
    n_edges = len(edge_i)

    degree      = np.zeros(n_nodes, dtype=np.float32)
    np.add.at(degree, edge_i, 1.0)
    d_inv_sqrt  = np.where(degree > 0, degree ** -0.5, 0.0)
    edge_weight = d_inv_sqrt[edge_i] * d_inv_sqrt[edge_j]

    indices = torch.from_numpy(np.vstack((edge_i, edge_j))).long()
    values  = torch.from_numpy(edge_weight)
    graph   = torch.sparse_coo_tensor(indices, values, (n_nodes, n_nodes)).coalesce()
    graph   = graph.to(CONFIG['device'])

    density = user_item_matrix.nnz / (n_users * n_items)
    print(f'[graph] {n_nodes:,} nodes ({n_users:,} users + {n_items:,} items)')
    print(f'  {n_edges:,} directed edges | interaction density: {density:.6f}')
    print(f'  Graph built in {time.time()-t0:.1f}s')
    return graph


graph = create_lightgcn_graph(train_matrix)


class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=128, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.embedding_user = nn.Embedding(n_users, emb_dim)
        self.embedding_item = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.embedding_user.weight, std=0.1)
        nn.init.normal_(self.embedding_item.weight, std=0.1)

    def forward(self, graph):
        all_emb = torch.cat([self.embedding_user.weight, self.embedding_item.weight])
        embs    = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(graph, all_emb)
            embs.append(all_emb)
        # Mean pooling qua các layer (kể cả layer 0)
        light_out = torch.mean(torch.stack(embs, dim=1), dim=1)
        return torch.split(light_out, [self.n_users, self.n_items])

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def bpr_loss(u_emb, pos_emb, neg_emb, u_emb0, pos_emb0, neg_emb0):
    pos_scores = (u_emb * pos_emb).sum(dim=1)
    neg_scores = (u_emb * neg_emb).sum(dim=1)
    bpr  = -F.logsigmoid(pos_scores - neg_scores).mean()
    reg  = (u_emb0.norm(2).pow(2) + pos_emb0.norm(2).pow(2) + neg_emb0.norm(2).pow(2))            / (2 * len(u_emb))
    return bpr, reg

[graph] 155,457 nodes (21,765 users + 133,692 items)
  10,121,248 directed edges | interaction density: 0.001755
  Graph built in 2.9s


## 4. Training với Weighted BPR

**Cải tiến chính:**

**Weighted Positive Sampling** — `WeightedRandomSampler` của PyTorch chọn (user, item) tỷ lệ với recency weight. Tương tác gần đây được học nhiều hơn.

**Popularity-based Negative Sampling** — precompute pool 2M samples theo `popularity^0.75`. Tránh gọi `np.random.choice(p=...)` mỗi `__getitem__` (O(n_items) ≈ O(133K) per call — cực chậm).

**CosineAnnealingLR** — LR giảm dần theo cosine từ `lr=1e-3` xuống `eta_min=1e-5`.

**Checkpoint** — lưu model mỗi 5 epoch và epoch cuối cùng.

In [9]:
class WeightedBPRDataset(Dataset):
    """
    Dataset BPR với precomputed negative pool.
    - Positive sampling: do WeightedRandomSampler bên ngoài quyết định
    - Negative sampling: popularity^0.75, precompute pool tránh O(n_items) per call
    """
    def __init__(self, user_item_matrix, neg_pool_size=2_000_000):
        coo = user_item_matrix.tocoo()
        self.users     = coo.row.astype(np.int32)
        self.pos_items = coo.col.astype(np.int32)
        self.weights   = coo.data.astype(np.float32)
        self.n_items   = user_item_matrix.shape[1]

        # ── Popularity-based negative distribution ────────────────────────────
        item_counts = np.asarray(user_item_matrix.sum(axis=0)).flatten()
        neg_probs   = np.power(item_counts + 1e-8, 0.75)
        neg_probs  /= neg_probs.sum()

        # ── Precompute pool: O(1) lookup thay vì O(n_items) per sample ────────
        print(f'  Precomputing neg pool ({neg_pool_size:,} samples)...')
        self._pool = np.random.choice(
            self.n_items, size=neg_pool_size, p=neg_probs
        ).astype(np.int32)
        self._pool_size = neg_pool_size

        top3 = np.bincount(self._pool[:100_000]).argsort()[-3:][::-1]
        print(f'  Top-3 negative items (most sampled): {top3.tolist()}')
        print(f'  Pos weight range: [{self.weights.min():.4f}, {self.weights.max():.2f}] '
              f'| mean={self.weights.mean():.4f}')

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = int(self.users[idx])
        i = int(self.pos_items[idx])
        # O(1): random index vào precomputed pool
        j = int(self._pool[np.random.randint(self._pool_size)])
        return u, i, j


# ── Khởi tạo model ────────────────────────────────────────────────────────────
n_users, n_items = train_matrix.shape
model = LightGCN(n_users, n_items,
                 emb_dim=CONFIG['emb_dim'],
                 n_layers=CONFIG['n_layers']).to(CONFIG['device'])

n_params = model.count_parameters()
print(f'[model] LightGCN({n_users:,} users, {n_items:,} items, dim={CONFIG["emb_dim"]}, layers={CONFIG["n_layers"]})')
print(f'  Trainable params: {n_params:,} ({n_params*4/1024/1024:.1f} MB fp32)')

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=1e-5
)

# ── DataLoader ────────────────────────────────────────────────────────────────
print('\n[dataset] Khởi tạo WeightedBPRDataset...')
dataset = WeightedBPRDataset(train_matrix, neg_pool_size=CONFIG['neg_pool_size'])

sampler = torch.utils.data.WeightedRandomSampler(
    weights     = torch.from_numpy(dataset.weights),
    num_samples = len(dataset),
    replacement = True
)
dataloader = DataLoader(
    dataset, batch_size=CONFIG['batch_size'],
    sampler=sampler, num_workers=2, pin_memory=(CONFIG['device']=='cuda')
)
print(f'  {len(dataset):,} samples | {len(dataloader)} batches/epoch | '
      f'batch_size={CONFIG["batch_size"]}')

# ── Training loop ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'Bắt đầu huấn luyện — {CONFIG["epochs"]} epochs')
print(f'{"="*60}')

train_history = []   # lưu loss theo epoch để debug
t_train_start = time.time()

for epoch in range(CONFIG['epochs']):
    model.train()
    total_bpr = total_reg = 0.0
    t_epoch   = time.time()

    for users, pos_items, neg_items in dataloader:
        users     = users.long().to(CONFIG['device'])
        pos_items = pos_items.long().to(CONFIG['device'])
        neg_items = neg_items.long().to(CONFIG['device'])

        optimizer.zero_grad()
        all_u, all_i = model(graph)

        u_emb    = all_u[users]
        pos_emb  = all_i[pos_items]
        neg_emb  = all_i[neg_items]
        u_emb0   = model.embedding_user(users)
        pos_emb0 = model.embedding_item(pos_items)
        neg_emb0 = model.embedding_item(neg_items)

        bpr, reg = bpr_loss(u_emb, pos_emb, neg_emb, u_emb0, pos_emb0, neg_emb0)
        (bpr + CONFIG['decay'] * reg).backward()

        # Gradient clipping — giữ reg loss không bùng nổ
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()

        total_bpr += bpr.item()
        total_reg += reg.item()

    scheduler.step()

    nb          = len(dataloader)
    avg_bpr     = total_bpr / nb
    avg_reg     = total_reg / nb
    current_lr  = optimizer.param_groups[0]['lr']
    epoch_time  = time.time() - t_epoch
    elapsed_tot = time.time() - t_train_start
    eta         = elapsed_tot / (epoch+1) * (CONFIG['epochs'] - epoch - 1)

    # ── Log embedding norm (giám sát reg loss) ────────────────────────────────
    with torch.no_grad():
        u_norm = model.embedding_user.weight.norm(dim=1).mean().item()
        i_norm = model.embedding_item.weight.norm(dim=1).mean().item()

    train_history.append({'epoch': epoch+1, 'bpr': avg_bpr, 'reg': avg_reg})
    print(f'Epoch {epoch+1:02d}/{CONFIG["epochs"]} | '
          f'BPR: {avg_bpr:.4f} | Reg: {avg_reg:.4f} | '
          f'|u|: {u_norm:.3f} | |i|: {i_norm:.3f} | '
          f'LR: {current_lr:.2e} | {epoch_time:.0f}s | ETA: {eta:.0f}s')

    # ── Checkpoint mỗi 15 epoch ────────────────────────────────────────────────
    if (epoch + 1) % 15 == 0 or (epoch + 1) == CONFIG['epochs']:
        ckpt = os.path.join(CONFIG['checkpoint_dir'], f'ckpt_epoch{epoch+1:02d}.pt')
        torch.save({
            'epoch'               : epoch + 1,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'bpr_loss'            : avg_bpr,
            'reg_loss'            : avg_reg,
            'config'              : CONFIG,
        }, ckpt)
        print(f'  ✅ Checkpoint saved → {ckpt}')

total_time = time.time() - t_train_start
print(f'\nTraining hoàn tất trong {total_time:.0f}s ({total_time/CONFIG["epochs"]:.1f}s/epoch)')

[model] LightGCN(21,765 users, 133,692 items, dim=128, layers=3)
  Trainable params: 19,898,496 (75.9 MB fp32)

[dataset] Khởi tạo WeightedBPRDataset...
  Precomputing neg pool (2,000,000 samples)...
  Top-3 negative items (most sampled): [15537, 26529, 17900]
  Pos weight range: [0.0000, 1527.43] | mean=0.6575
  5,108,014 samples | 624 batches/epoch | batch_size=8192

Bắt đầu huấn luyện — 20 epochs
Epoch 01/20 | BPR: 0.5426 | Reg: 4.4472 | |u|: 2.356 | |i|: 1.847 | LR: 9.94e-04 | 414s | ETA: 7867s
Epoch 02/20 | BPR: 0.3344 | Reg: 10.9856 | |u|: 3.214 | |i|: 2.255 | LR: 9.76e-04 | 420s | ETA: 7506s
Epoch 03/20 | BPR: 0.2740 | Reg: 15.0458 | |u|: 3.769 | |i|: 2.515 | LR: 9.46e-04 | 420s | ETA: 7106s
Epoch 04/20 | BPR: 0.2408 | Reg: 17.8827 | |u|: 4.196 | |i|: 2.717 | LR: 9.05e-04 | 420s | ETA: 6695s
Epoch 05/20 | BPR: 0.2185 | Reg: 20.0489 | |u|: 4.542 | |i|: 2.883 | LR: 8.55e-04 | 420s | ETA: 6281s
Epoch 06/20 | BPR: 0.2013 | Reg: 21.8350 | |u|: 4.835 | |i|: 3.025 | LR: 7.96e-04 | 420s

## 5. Trích Xuất Embeddings & Xây Dựng FAISS Index

In [10]:
t0 = time.time()
model.eval()
with torch.no_grad():
    print('[embed] Đang trích xuất final embeddings từ đồ thị...')
    final_u, final_i = model(graph)
    user_vectors = final_u.cpu().numpy().astype(np.float32)
    item_vectors = final_i.cpu().numpy().astype(np.float32)

# L2-normalize trước khi nạp vào FAISS (IndexFlatIP → cosine similarity)
faiss.normalize_L2(user_vectors)
faiss.normalize_L2(item_vectors)
print(f'  user_vectors: {user_vectors.shape} | norm sample: {np.linalg.norm(user_vectors[0]):.4f}')
print(f'  item_vectors: {item_vectors.shape} | norm sample: {np.linalg.norm(item_vectors[0]):.4f}')

# ── Xây dựng FAISS Index ──────────────────────────────────────────────────────
print('\n[faiss] Xây dựng IndexFlatIP...')
dim   = CONFIG['emb_dim']
index = faiss.IndexFlatIP(dim)

if torch.cuda.is_available() and hasattr(faiss, 'StandardGpuResources'):
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)
    print('  FAISS → GPU')
else:
    print('  FAISS → CPU')

index.add(item_vectors)
print(f'  Đã nạp {index.ntotal:,} items | Build time: {time.time()-t0:.1f}s')

# ── Sanity check: query 1 user ────────────────────────────────────────────────
D, I = index.search(user_vectors[:1], 5)
print(f'  Sanity check user[0] → top-5 scores: {D[0].round(4).tolist()}')

# ── Lưu artifacts ─────────────────────────────────────────────────────────────
print('\n[save] Lưu artifacts...')
D_dir = CONFIG['output_dir']

np.save(os.path.join(D_dir, 'user_vectors.npy'), user_vectors)
np.save(os.path.join(D_dir, 'item_vectors.npy'), item_vectors)

mappings = {
    'user2idx'            : accumulator.user2idx,
    'item2idx'            : accumulator.item2idx,
    'idx2user'            : accumulator.idx2user,
    'idx2item'            : accumulator.idx2item,
    'item_meta'           : accumulator.item_meta,
    'user_raw_items'      : dict(accumulator.user_raw_items),
    'user_item_ts_matrix' : dict(accumulator.user_item_ts_matrix),
    'global_max_ts'       : accumulator.global_max_ts,
    'config'              : CONFIG,
}
joblib.dump(mappings, os.path.join(D_dir, 'index_mappings.pkl'))

sp.save_npz(os.path.join(D_dir, 'train_user_item.npz'), train_matrix)
sp.save_npz(os.path.join(D_dir, 'test_user_item.npz'),  test_matrix)

# ── In kích thước file ────────────────────────────────────────────────────────
for fname in ['user_vectors.npy','item_vectors.npy','index_mappings.pkl',
              'train_user_item.npz','test_user_item.npz']:
    fpath = os.path.join(D_dir, fname)
    mb    = os.path.getsize(fpath) / 1024 / 1024
    print(f'  {fname:<30s} {mb:6.1f} MB')
print('✅ Lưu hoàn tất.')

[embed] Đang trích xuất final embeddings từ đồ thị...
  user_vectors: (21765, 128) | norm sample: 1.0000
  item_vectors: (133692, 128) | norm sample: 1.0000

[faiss] Xây dựng IndexFlatIP...
  FAISS → CPU
  Đã nạp 133,692 items | Build time: 0.4s
  Sanity check user[0] → top-5 scores: [0.9021000266075134, 0.8981999754905701, 0.8981000185012817, 0.897599995136261, 0.8934999704360962]

[save] Lưu artifacts...
  user_vectors.npy                 10.6 MB
  item_vectors.npy                 65.3 MB
  index_mappings.pkl             1527.5 MB
  train_user_item.npz              27.5 MB
  test_user_item.npz                7.1 MB
✅ Lưu hoàn tất.


## 6. Lớp Gợi Ý Toàn Diện

**Bug fixes:**
- `if uid is None` — uid=0 (user đầu tiên) không còn bị nhầm là cold-start
- `generate_playlist`: normalize vector trước FAISS search, log seed tracks tìm được
- `recommend_by_timeframe`: không cần `raw_history_df`, dùng timestamps đã lưu

**Thêm mới:**
- Log chi tiết trong mỗi hàm
- `evaluate_metrics`: tqdm progress + NDCG@K

In [11]:
class LightGCNAdvancedRecommender:

    def __init__(self, model_dir: str):
        t0 = time.time()
        print(f'[recommender] Loading từ {model_dir}...')
        self.user_vectors = np.load(os.path.join(model_dir, 'user_vectors.npy'))
        self.item_vectors = np.load(os.path.join(model_dir, 'item_vectors.npy'))
        print(f'  Embeddings: user={self.user_vectors.shape}, item={self.item_vectors.shape}')

        m = joblib.load(os.path.join(model_dir, 'index_mappings.pkl'))
        self.user2idx            = m['user2idx']
        self.item2idx            = m['item2idx']
        self.idx2user            = m['idx2user']
        self.idx2item            = m['idx2item']
        self.item_meta           = m['item_meta']
        self.user_raw_items      = m['user_raw_items']
        self.user_item_ts_matrix = m.get('user_item_ts_matrix', {})
        self.global_max_ts       = m.get('global_max_ts', 0.0)
        self._cfg                = m.get('config', {})

        print(f'  {len(self.user2idx):,} users | {len(self.item2idx):,} items | '
              f'{len(self.item_meta):,} item metadata')

        self.train_matrix = sp.load_npz(os.path.join(model_dir, 'train_user_item.npz'))
        self.test_matrix  = sp.load_npz(os.path.join(model_dir, 'test_user_item.npz'))

        self.dim   = self.item_vectors.shape[1]
        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(self.item_vectors)
        print(f'  FAISS index: {self.index.ntotal:,} items (dim={self.dim})')
        print(f'✅ Recommender sẵn sàng ({time.time()-t0:.1f}s)')

    # ──────────────────────────── Helpers ─────────────────────────────────────
    def _to_row(self, msid, score):
        meta = self.item_meta.get(msid, {})
        return {'track_name' : meta.get('track_name',  msid),
                'artist_name': meta.get('artist_name', ''),
                'score'      : round(float(score), 4)}

    def _normalize(self, vec: np.ndarray) -> np.ndarray:
        """L2-normalize: copy an toàn, reshape 2-D, inplace normalize."""
        v = vec.copy().reshape(1, -1).astype(np.float32)
        faiss.normalize_L2(v)
        return v

    def _mmr_rerank(self, rec_ids, scores, n=10, lambda_=0.6):
        """Maximal Marginal Relevance — đa dạng hóa danh sách gợi ý."""
        if len(rec_ids) == 0:
            return [], []
        rec_ids   = np.array(rec_ids)
        scores    = np.array(scores)
        norm_sc   = scores / (scores.max() + 1e-8)
        factors   = self.item_vectors[rec_ids]
        sim_mat   = factors @ factors.T

        selected, unsel = [], list(range(len(rec_ids)))
        first = int(np.argmax(norm_sc))
        selected.append(first); unsel.remove(first)

        while len(selected) < n and unsel:
            rel        = norm_sc[unsel]
            sim        = sim_mat[np.ix_(unsel, selected)].max(axis=1)
            mmr_scores = lambda_ * rel - (1 - lambda_) * sim
            best_local = int(np.argmax(mmr_scores))
            selected.append(unsel[best_local])
            unsel.pop(best_local)

        return rec_ids[selected].tolist(), scores[selected].tolist()

    def _listened_set(self, user_id_str):
        raw = self.user_raw_items.get(str(user_id_str), set())
        return {self.item2idx[m] for m in raw if m in self.item2idx}

    def popular_items(self, n=10):
        counts  = np.asarray(self.train_matrix.sum(axis=0)).flatten()
        top_ids = np.argsort(counts)[::-1][:n]
        df = pd.DataFrame([self._to_row(self.idx2item[i], counts[i]) for i in top_ids])
        df.index = range(1, len(df)+1)
        return df

    # ──────────────────────── Feature 1: Gợi ý cơ bản ────────────────────────
    def recommend(self, user_id_str, n=10, filter_listened=True, use_mmr=True):
        uid = self.user2idx.get(str(user_id_str))

        if uid is None:   # FIX: `is None` không phải `not uid`
            raw_msids = self.user_raw_items.get(str(user_id_str), set())
            print(f'  [recommend] Cold-start user | {len(raw_msids)} bài đã biết')
            if len(raw_msids) < 3:
                print('  → Ít hơn 3 bài, trả về popular items')
                return self.popular_items(n)
            listened_iids = [self.item2idx[m] for m in raw_msids if m in self.item2idx]
            user_vec = self._normalize(np.mean(self.item_vectors[listened_iids], axis=0))
            print(f'  → Item-based proxy vector từ {len(listened_iids)} bài')
        else:
            n_train = self.train_matrix[uid].nnz
            n_test  = self.test_matrix[uid].nnz
            print(f'  [recommend] uid={uid} | train={n_train} bài | test={n_test} bài')
            user_vec = self.user_vectors[uid:uid+1]

        scores, indices = self.index.search(user_vec, n * 5)
        listened = self._listened_set(user_id_str)

        rec_ids, rec_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            if filter_listened and iid in listened:
                continue
            rec_ids.append(int(iid)); rec_scores.append(float(sc))

        n_candidates = len(rec_ids)
        if use_mmr:
            rec_ids, rec_scores = self._mmr_rerank(rec_ids, rec_scores, n=n)
            print(f'  → {n_candidates} candidates → MMR rerank → top {len(rec_ids)}')
        else:
            rec_ids, rec_scores = rec_ids[:n], rec_scores[:n]

        df = pd.DataFrame([self._to_row(self.idx2item[i], s) for i, s in zip(rec_ids, rec_scores)])
        df.index = range(1, len(df)+1)
        return df

    # ──────────────────── Feature 2: Tạo Playlist theo ngữ cảnh ──────────────
    def generate_playlist(self, user_id_str, seed_track_names=None,
                          playlist_name='Gợi ý dành riêng cho bạn', n_songs=15):
        print(f'\n[playlist] "{playlist_name}"')
        uid = self.user2idx.get(str(user_id_str))

        if uid is None:   # FIX: is None
            print('  Cold-start user → popular items')
            return self.popular_items(n_songs)

        base_vec = self.user_vectors[uid].copy()   # 1-D, đã normalize

        if seed_track_names:
            found, not_found = [], []
            seed_vecs = []
            for track in seed_track_names:
                t_lower = track.lower()
                hit = None
                for msid, meta in self.item_meta.items():
                    if meta.get('track_name','').lower() == t_lower and msid in self.item2idx:
                        hit = msid; break
                if hit:
                    seed_vecs.append(self.item_vectors[self.item2idx[hit]])
                    found.append(track)
                else:
                    not_found.append(track)

            if found:
                print(f'  Seed tracks tìm thấy  : {found}')
            if not_found:
                print(f'  ⚠ Seed tracks không tìm thấy: {not_found} (catalog chỉ chứa items đã lọc)')

            if seed_vecs:
                seed_mean = np.mean(seed_vecs, axis=0)
                seed_norm = seed_mean / (np.linalg.norm(seed_mean) + 1e-8)
                base_vec  = 0.7 * base_vec + 0.3 * seed_norm   # 70% gu user, 30% seed
                print(f'  Blend: 70% user profile + 30% seed ({len(seed_vecs)} tracks)')
            else:
                print('  Không có seed khả dụng → dùng thuần user profile')

        query_vec = self._normalize(base_vec)   # FIX: normalize TRƯỚC khi search
        scores, indices = self.index.search(query_vec, n_songs * 4)
        listened = self._listened_set(user_id_str)

        rec_ids, rec_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            if iid in listened: continue
            rec_ids.append(int(iid)); rec_scores.append(float(sc))

        final_ids, final_scores = self._mmr_rerank(rec_ids, rec_scores, n=n_songs, lambda_=0.5)
        print(f'  → {len(rec_ids)} candidates → MMR (λ=0.5) → {len(final_ids)} bài')
        df = pd.DataFrame([self._to_row(self.idx2item[i], s) for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df)+1)
        return df

    # ──────────────────── Feature 3: Real-time Late Fusion ────────────────────
    def recommend_realtime(self, user_id_str, recent_listened_msids, n=10, alpha=0.4):
        print(f'\n[realtime] User {user_id_str} | α={alpha} (short-term weight)')
        uid = self.user2idx.get(str(user_id_str))

        found_msids = [m for m in recent_listened_msids if m in self.item2idx]
        print(f'  Recent tracks: {len(recent_listened_msids)} nhập vào | {len(found_msids)} có trong catalog')

        if not found_msids:
            print('  Không có recent track khả dụng → fallback recommend()')
            return self.recommend(user_id_str, n=n)

        short_vec = np.mean([self.item_vectors[self.item2idx[m]] for m in found_msids], axis=0)

        if uid is not None:
            long_vec = self.user_vectors[uid]
            fused    = (1 - alpha) * long_vec + alpha * short_vec
            print(f'  Fused: {1-alpha:.0%} long-term (uid={uid}) + {alpha:.0%} short-term')
        else:
            fused = short_vec
            print('  Cold-start → pure short-term vector')

        query_vec = self._normalize(fused)
        scores, indices = self.index.search(query_vec, n * 4)

        listened   = self._listened_set(user_id_str)
        recent_set = {self.item2idx[m] for m in found_msids}
        exclude    = listened | recent_set

        rec_ids, rec_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            if iid in exclude: continue
            rec_ids.append(int(iid)); rec_scores.append(float(sc))

        final_ids, final_scores = self._mmr_rerank(rec_ids, rec_scores, n=n)
        df = pd.DataFrame([self._to_row(self.idx2item[i], s) for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df)+1)
        return df

    # ──────────────────── Feature 4: Time-bound (không cần raw DF) ────────────
    def recommend_by_timeframe(self, user_id_str, start_date, end_date, n=10):
        print(f'\n[timeframe] User {user_id_str} | {start_date} → {end_date}')
        uid = self.user2idx.get(str(user_id_str))
        if uid is None:
            print('  User không có trong model → popular items')
            return self.popular_items(n)

        start_ts = pd.Timestamp(start_date).timestamp()
        end_ts   = pd.Timestamp(end_date).timestamp()
        ts_dict  = self.user_item_ts_matrix.get(uid, {})

        if not ts_dict:
            print('  ⚠ User không có timestamp data trong model')
            return pd.DataFrame()

        # Toàn bộ lịch sử của user
        all_ts = list(ts_dict.values())
        user_range = (pd.Timestamp(min(all_ts), unit='s').date(),
                      pd.Timestamp(max(all_ts), unit='s').date())
        print(f'  User lịch sử: {user_range[0]} → {user_range[1]} ({len(ts_dict)} bài)')

        period_iids = [iid for iid, ts in ts_dict.items() if start_ts <= ts <= end_ts]
        if not period_iids:
            print(f'  ❌ Không có bài nào trong giai đoạn này')
            return pd.DataFrame()

        print(f'  Tìm thấy {len(period_iids)} bài trong giai đoạn')

        # Weighted mean — bài nghe gần cuối period được ưu tiên hơn
        ts_arr = np.array([ts_dict[iid] for iid in period_iids])
        w      = np.exp(-(ts_arr.max() - ts_arr) / (30 * 86400))   # 30-day HL
        w     /= w.sum()
        period_vec = self._normalize((self.item_vectors[period_iids] * w[:, None]).sum(axis=0))

        scores, indices = self.index.search(period_vec, n * 3)
        period_set = set(period_iids)

        rec_ids, rec_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            if iid in period_set: continue
            rec_ids.append(int(iid)); rec_scores.append(float(sc))

        final_ids, final_scores = self._mmr_rerank(rec_ids, rec_scores, n=n)
        df = pd.DataFrame([self._to_row(self.idx2item[i], s) for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df)+1)
        return df

    # ──────────────────── Feature 5: Evaluate Recall, Precision, NDCG ─────────
    def evaluate_metrics(self, K=20, eval_batch=512):
        print(f'\n[evaluate] K={K} | test set = bài nghe gần nhất (temporal split)')
        test_users = np.unique(self.test_matrix.nonzero()[0])
        n_users    = len(test_users)
        print(f'  Đánh giá trên {n_users:,} users...')

        total_recall = total_precision = total_ndcg = 0.0
        t0 = time.time()

        for start in tqdm(range(0, n_users, eval_batch), desc='Evaluating', unit='batch'):
            batch_u    = test_users[start:start + eval_batch]
            batch_vecs = self.user_vectors[batch_u]
            _, indices = self.index.search(batch_vecs, K * 3)

            for i, uid in enumerate(batch_u):
                actual     = set(self.test_matrix[uid].indices)
                if not actual: continue
                train_seen = set(self.train_matrix[uid].indices)

                preds = []
                for iid in indices[i]:
                    if iid not in train_seen:
                        preds.append(iid)
                    if len(preds) == K:
                        break

                hits      = actual & set(preds)
                hit_count = len(hits)
                total_recall    += hit_count / len(actual)
                total_precision += hit_count / K

                # NDCG@K
                dcg  = sum(1.0 / math.log2(r + 2) for r, p in enumerate(preds) if p in actual)
                idcg = sum(1.0 / math.log2(r + 2) for r in range(min(len(actual), K)))
                total_ndcg += dcg / idcg if idcg > 0 else 0.0

        recall    = total_recall    / n_users
        precision = total_precision / n_users
        ndcg      = total_ndcg      / n_users

        print(f'\n  ┌── Kết quả ({time.time()-t0:.1f}s) ──────────┐')
        print(f'  │  Recall@{K:<3}    : {recall:.4f}              │')
        print(f'  │  Precision@{K:<3} : {precision:.4f}              │')
        print(f'  │  NDCG@{K:<3}      : {ndcg:.4f}              │')
        print(f'  └────────────────────────────────────┘')
        return recall, precision, ndcg


rec_sys = LightGCNAdvancedRecommender(CONFIG['output_dir'])

[recommender] Loading từ /kaggle/working/lightgcn_model...
  Embeddings: user=(21765, 128), item=(133692, 128)
  21,765 users | 133,692 items | 11,261,940 item metadata
  FAISS index: 133,692 items (dim=128)
✅ Recommender sẵn sàng (115.3s)


## 7. Demo Toàn Bộ Chức Năng

In [12]:
# ── Chọn user demo ───────────────────────────────────────────────────────────
test_uid_idx = np.unique(rec_sys.test_matrix.nonzero()[0])[0]
test_user    = rec_sys.idx2user[test_uid_idx]

print('=' * 55)
print(f'  Demo User : {test_user}')
print(f'  Tổng bài đã nghe : {len(rec_sys.user_raw_items.get(test_user, set())):,}')
print(f'  Train               : {rec_sys.train_matrix[test_uid_idx].nnz} bài')
print(f'  Test (gần nhất)     : {rec_sys.test_matrix[test_uid_idx].nnz} bài')
# Hiển thị khoảng thời gian lịch sử nếu có
uid_ts = rec_sys.user_item_ts_matrix.get(test_uid_idx, {})
if uid_ts:
    ts_vals = list(uid_ts.values())
    print(f'  Timestamp range     : '
          f'{pd.Timestamp(min(ts_vals), unit="s").date()} → '
          f'{pd.Timestamp(max(ts_vals), unit="s").date()}')
print('=' * 55)

# ── 1. Evaluation ─────────────────────────────────────────────────────────────
rec_sys.evaluate_metrics(K=20)

# ── 2. Gợi ý cơ bản ──────────────────────────────────────────────────────────
print(f'\n──── Top 10 gợi ý (FAISS + MMR) ────')
display(rec_sys.recommend(test_user, n=10, use_mmr=True))

# ── 3. Playlist ───────────────────────────────────────────────────────────────
display(rec_sys.generate_playlist(
    user_id_str     = test_user,
    seed_track_names= ['Lightbulb Sun', 'Loner'],
    playlist_name   = 'Rock/Metal Playlist',
    n_songs         = 10
))

# ── 4. Real-time ──────────────────────────────────────────────────────────────
recent_msids = list(rec_sys.item2idx.keys())[:3]
display(rec_sys.recommend_realtime(
    test_user, recent_listened_msids=recent_msids, n=10, alpha=0.5
))

# ── 5. Time-bound ─────────────────────────────────────────────────────────────
if rec_sys.global_max_ts > 0 and uid_ts:
    end_dt   = pd.Timestamp(rec_sys.global_max_ts, unit='s')
    start_dt = end_dt - pd.Timedelta(days=180)
    display(rec_sys.recommend_by_timeframe(
        test_user,
        start_date = str(start_dt.date()),
        end_date   = str(end_dt.date()),
        n          = 10
    ))
else:
    print('\n⚠ Bỏ qua demo time-bound (không có timestamp data)')

  Demo User : 13
  Tổng bài đã nghe : 2,576
  Train               : 180 bài
  Test (gần nhất)     : 44 bài
  Timestamp range     : 2026-02-06 → 2026-03-09

[evaluate] K=20 | test set = bài nghe gần nhất (temporal split)
  Đánh giá trên 18,864 users...


Evaluating: 100%|██████████| 37/37 [00:10<00:00,  3.47batch/s]


  ┌── Kết quả (10.7s) ──────────┐
  │  Recall@20     : 0.0415              │
  │  Precision@20  : 0.0320              │
  │  NDCG@20       : 0.0499              │
  └────────────────────────────────────┘

──── Top 10 gợi ý (FAISS + MMR) ────
  [recommend] uid=0 | train=180 bài | test=44 bài
  → 46 candidates → MMR rerank → top 10


,track_name,artist_name,score
1,Things Left Unsaid,Pink Floyd,0.8982
2,Hush Mail,Infected Mushroom,0.8834
3,Release Me,Infected Mushroom,0.8976
4,We Built Our Own World,Hans Zimmer,0.8368
5,Omen (Reprise),The Prodigy,0.8795
6,Jaded,The Crystal Method,0.8712
7,Conga Fury,Juno Reactor,0.8479
8,Dream On,Depeche Mode,0.8353
9,Blade Runner (End Titles),Vangelis,0.8390
10,It's What We Do,Pink Floyd,0.8981



[playlist] "Rock/Metal Playlist"
  Seed tracks tìm thấy  : ['Lightbulb Sun', 'Loner']
  Blend: 70% user profile + 30% seed (2 tracks)
  → 34 candidates → MMR (λ=0.5) → 10 bài


,track_name,artist_name,score
1,It's What We Do,Pink Floyd,0.9061
2,Orion,Rodrigo y Gabriela,0.8687
3,Shot in the Back of the Head,Moby,0.8745
4,True Grit,The Crystal Method,0.8739
5,The End of the Beginning,God Is an Astronaut,0.8840
6,Well Worn Hand,Editors,0.8728
7,Then Came the Last Days of May,Blue Öyster Cult,0.8698
8,Smile?,The Crystal Method,0.8769
9,Surrender,The Chemical Brothers,0.8843
10,Escape Velocity,The Chemical Brothers,0.8976



[realtime] User 13 | α=0.5 (short-term weight)
  Recent tracks: 3 nhập vào | 3 có trong catalog
  Fused: 50% long-term (uid=0) + 50% short-term


,track_name,artist_name,score
1,We Are the Night,The Chemical Brothers,0.9116
2,Almost June,Ludovico Einaudi,0.8892
3,Overstand,Thievery Corporation,0.8796
4,Trip Like I Do,The Crystal Method,0.8928
5,Blade Runner (End Titles),Vangelis,0.8851
6,Always Something Better,Trentemøller,0.8874
7,In Chains,Depeche Mode,0.8828
8,Surrender,The Chemical Brothers,0.8982
9,Escape Velocity,The Chemical Brothers,0.9031
10,Get Up Get Off,The Prodigy,0.8797



[timeframe] User 13 | 2025-09-11 → 2026-03-10
  User lịch sử: 2026-02-06 → 2026-03-09 (224 bài)
  Tìm thấy 224 bài trong giai đoạn


,track_name,artist_name,score
1,Useless,Depeche Mode,0.9229
2,Keep Hope Alive,The Crystal Method,0.9018
3,Orion,Rodrigo y Gabriela,0.9041
4,Surrender,The Chemical Brothers,0.9181
5,Ebb and Flow,Pink Floyd,0.9163
6,Escape Velocity,The Chemical Brothers,0.9136
7,So Lonely,The Police,0.9179
8,Omen (Reprise),The Prodigy,0.9141
9,Their Law,The Prodigy,0.9082
10,Nobody's Fault but Mine,Led Zeppelin,0.9159


## 8. Hybrid Cold-Start + Enhanced Inference Layer

### Cold-Start Tier Strategy

| Tier | Condition | Strategy |
|------|-----------|----------|
| **warm** | >= `COLD_THRESHOLD` interactions | Pure LightGCN embedding |
| **lukewarm** | 1...threshold-1 interactions | Blend: alpha*LightGCN + (1-alpha)*item-proxy |
| **cold** | 0 interactions / new user | Content-based proxy (TF-IDF+SVD) or Popular |
| **new_item** | Item not in catalog | TF-IDF content similarity |

### Additional Inference Strategies

| Method | Description |
|--------|-------------|
| `recommend_hybrid` | Unified entry point, auto-detects tier |
| `recommend_similar_to_new_item` | Cold-start item via content similarity |
| `recommend_similar_users` | User-user CF (k nearest neighbors) |
| `recommend_trending` | Popularity x recency decay, optionally blended with user pref |
| `recommend_discovery` | Serendipity: blend with far-user embeddings |
| `recommend_next_in_session` | Session-based next-track prediction (exp decay) |
| `recommend_by_artist` | By artist + expand to similar artists via embeddings |


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from collections import defaultdict

COLD_THRESHOLD = 5


class HybridRecommender(LightGCNAdvancedRecommender):
    def __init__(self, model_dir: str, cold_threshold: int = COLD_THRESHOLD):
        super().__init__(model_dir)
        self.cold_threshold = cold_threshold
        self._build_content_index()
        self._build_artist_index()
        self._build_trending_scores()
        self._user_index = None
        print(f'[HybridRecommender] Ready | cold_threshold={self.cold_threshold}')

    # ── A. BUILD INDEXES ──────────────────────────────────────────────────────

    def _build_content_index(self, svd_dim: int = 64):
        """TF-IDF char-ngram(2-4) -> TruncatedSVD -> normalize -> FAISS."""
        t0 = time.time()
        texts, iids = [], []
        for msid, meta in self.item_meta.items():
            if msid in self.item2idx:
                text = (meta.get('track_name', '') + ' ' +
                        meta.get('artist_name', '')).strip().lower()
                texts.append(text)
                iids.append(self.item2idx[msid])

        self._content_iids = np.array(iids, dtype=np.int32)

        self._tfidf = TfidfVectorizer(
            analyzer='char_wb', ngram_range=(2, 4),
            max_features=30_000, sublinear_tf=True, min_df=2,
        )
        tfidf_mat = self._tfidf.fit_transform(texts)

        real_dim = min(svd_dim, tfidf_mat.shape[1] - 1, tfidf_mat.shape[0] - 1)
        self._svd = TruncatedSVD(n_components=real_dim, random_state=42)
        dense = self._svd.fit_transform(tfidf_mat).astype(np.float32)
        faiss.normalize_L2(dense)

        self._content_dim   = real_dim
        self._content_vecs  = dense
        self._content_index = faiss.IndexFlatIP(real_dim)
        self._content_index.add(dense)
        self._iid_to_content_pos = {int(iid): pos for pos, iid in enumerate(self._content_iids)}

        exp_var = self._svd.explained_variance_ratio_.sum()
        print(f'[content_index] {len(texts):,} items | vocab={tfidf_mat.shape[1]:,} | '
              f'SVD({real_dim}) expl_var={exp_var:.2%} | {time.time()-t0:.1f}s')

    def _build_artist_index(self):
        """artist -> list[item_idx]; artist_emb = mean(LightGCN item vecs) -> FAISS."""
        t0 = time.time()
        artist2items: dict = defaultdict(list)
        self._iid_to_artist: dict = {}

        for msid, meta in self.item_meta.items():
            if msid in self.item2idx:
                artist = meta.get('artist_name', '').strip().lower()
                iid    = self.item2idx[msid]
                if artist:
                    artist2items[artist].append(iid)
                    self._iid_to_artist[iid] = artist

        self._artist2items       = dict(artist2items)
        self._artist_names       = list(artist2items.keys())
        self._artist_name_to_pos = {name: pos for pos, name in enumerate(self._artist_names)}

        artist_vecs = np.array(
            [self.item_vectors[items].mean(axis=0) for items in artist2items.values()],
            dtype=np.float32,
        )
        faiss.normalize_L2(artist_vecs)
        self._artist_vecs  = artist_vecs
        self._artist_index = faiss.IndexFlatIP(self.dim)
        self._artist_index.add(artist_vecs)
        print(f'[artist_index] {len(self._artist2items):,} artists | {time.time()-t0:.1f}s')

    def _build_trending_scores(self, halflife_days: int = 30):
        """trending = popularity_count * exp(-days_since_avg_listen / hl), norm [0,1]."""
        t0  = time.time()
        n   = self.item_vectors.shape[0]
        pop = np.asarray(self.train_matrix.sum(axis=0)).flatten().astype(np.float64)

        ts_sum   = np.zeros(n, dtype=np.float64)
        ts_count = np.zeros(n, dtype=np.int32)
        for uid_key, ts_dict in self.user_item_ts_matrix.items():
            for iid, ts in ts_dict.items():
                iid_int = int(iid)
                if 0 <= iid_int < n:
                    ts_sum[iid_int]   += float(ts)
                    ts_count[iid_int] += 1

        with np.errstate(invalid='ignore', divide='ignore'):
            avg_ts = np.where(ts_count > 0, ts_sum / np.maximum(ts_count, 1),
                              float(self.global_max_ts))

        hl_sec   = halflife_days * 86400.0
        days_ago = np.clip((self.global_max_ts - avg_ts) / (hl_sec + 1e-9), 0, None)
        trending  = pop * np.exp(-days_ago)
        t_max     = trending.max()
        self._trending_scores = (
            (trending / t_max).astype(np.float32) if t_max > 0
            else trending.astype(np.float32)
        )
        print(f'[trending] top={self._trending_scores.max():.4f} | '
              f'mean={self._trending_scores.mean():.4f} | {time.time()-t0:.1f}s')

    # ── B. HELPERS ────────────────────────────────────────────────────────────

    def _get_user_tier(self, user_id_str: str):
        uid = self.user2idx.get(str(user_id_str))
        if uid is None:
            return 'cold', None
        n = self.train_matrix[uid].nnz
        return ('warm' if n >= self.cold_threshold else 'lukewarm'), uid

    def _proxy_vector_from_items(self, iids, weights=None) -> np.ndarray:
        iids = np.asarray(iids, dtype=np.int32)
        vecs = self.item_vectors[iids].astype(np.float32)
        if weights is not None:
            w = np.asarray(weights, dtype=np.float32)
            w = w / (w.sum() + 1e-12)
            vec = (vecs * w[:, None]).sum(axis=0, keepdims=True)
        else:
            vec = vecs.mean(axis=0, keepdims=True)
        return self._normalize(vec)

    def _text_to_content_vec(self, text: str) -> np.ndarray:
        v = self._svd.transform(
            self._tfidf.transform([text.strip().lower()])
        ).astype(np.float32)
        faiss.normalize_L2(v)
        return v

    def _apply_artist_diversity(self, rec_ids: list, rec_scores: list,
                                 artist_limit: int = 2):
        artist_count: dict = defaultdict(int)
        f_ids, f_scores = [], []
        for iid, sc in zip(rec_ids, rec_scores):
            artist = self._iid_to_artist.get(int(iid), f'__unk_{iid}')
            if artist_count[artist] < artist_limit:
                f_ids.append(iid); f_scores.append(sc)
                artist_count[artist] += 1
        return f_ids, f_scores

    def _cold_user_vector(self, liked_tracks=None, liked_artists=None,
                          user_id_str=None):
        proxy_iids: list = []

        if liked_tracks:
            for track in liked_tracks:
                cvec = self._text_to_content_vec(track)
                _, cpos = self._content_index.search(cvec, 5)
                proxy_iids.extend(self._content_iids[cpos[0]].tolist())

        if liked_artists:
            for artist in liked_artists:
                a_lower = artist.strip().lower()
                if a_lower in self._artist2items:
                    proxy_iids.extend(self._artist2items[a_lower][:10])
                else:
                    cvec = self._text_to_content_vec(a_lower)
                    _, cpos = self._content_index.search(cvec, 5)
                    proxy_iids.extend(self._content_iids[cpos[0]].tolist())

        if user_id_str and not proxy_iids:
            raw = self.user_raw_items.get(str(user_id_str), set())
            proxy_iids.extend(self.item2idx[m] for m in raw if m in self.item2idx)

        if not proxy_iids:
            return None

        seen, deduped = set(), []
        for x in proxy_iids:
            if x not in seen:
                seen.add(x); deduped.append(x)

        print(f'  Cold proxy: {len(deduped)} anchor items from text hints')
        return self._proxy_vector_from_items(deduped)

    def _ensure_user_index(self):
        if self._user_index is None:
            print('  [lazy] Building user FAISS index...')
            self._user_index = faiss.IndexFlatIP(self.dim)
            self._user_index.add(self.user_vectors)

    # ── 1. HYBRID RECOMMEND (main entry point) ────────────────────────────────

    def recommend_hybrid(self, user_id_str, n: int = 10,
                         liked_tracks=None, liked_artists=None,
                         filter_listened: bool = True, use_mmr: bool = True,
                         artist_limit: int = 2,
                         trending_boost: float = 0.0) -> pd.DataFrame:
        """
        Unified entry point -- auto-detect tier and choose strategy:
          warm     : pure LightGCN embedding
          lukewarm : blend LightGCN (alpha) + item-mean proxy (1-alpha)
          cold     : content-based proxy from liked_tracks / liked_artists

        Parameters
        ----------
        liked_tracks   : list[str] -- track names user likes (for cold-start)
        liked_artists  : list[str] -- artist names user likes (for cold-start)
        artist_limit   : int -- max songs per artist in output (0 = off)
        trending_boost : float [0,1] -- blend trending score into final score
        """
        tier, uid = self._get_user_tier(user_id_str)
        print(f'[hybrid] User={user_id_str!r} | tier={tier} | uid={uid}')

        if tier == 'warm':
            query_vec = self.user_vectors[uid:uid+1].copy()
            print(f'  LightGCN embedding (uid={uid}, '
                  f'{self.train_matrix[uid].nnz} interactions)')

        elif tier == 'lukewarm':
            n_inter   = self.train_matrix[uid].nnz
            lgcn_vec  = self.user_vectors[uid].copy()
            item_iids = list(self.train_matrix[uid].indices)
            proxy_vec = self._proxy_vector_from_items(item_iids).flatten()
            alpha     = min(0.5 + 0.5 * (n_inter / max(self.cold_threshold, 1)), 1.0)
            fused     = alpha * lgcn_vec + (1.0 - alpha) * proxy_vec
            query_vec = self._normalize(fused)
            print(f'  Hybrid lukewarm: {alpha:.0%} LightGCN + '
                  f'{1-alpha:.0%} item-proxy ({n_inter} interactions)')

        else:  # cold
            query_vec = self._cold_user_vector(liked_tracks, liked_artists, user_id_str)
            if query_vec is None:
                print('  -> insufficient info -> popular items')
                return self.popular_items(n)

        scores, indices = self.index.search(query_vec, n * 8)
        listened = self._listened_set(user_id_str) if filter_listened else set()

        rec_ids, rec_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            if filter_listened and iid in listened:
                continue
            if trending_boost > 0.0:
                sc = ((1.0 - trending_boost) * float(sc)
                      + trending_boost * float(self._trending_scores[iid]))
            rec_ids.append(int(iid)); rec_scores.append(float(sc))

        if artist_limit > 0:
            rec_ids, rec_scores = self._apply_artist_diversity(
                rec_ids, rec_scores, artist_limit)

        n_cand = len(rec_ids)
        if use_mmr and n_cand >= n:
            final_ids, final_scores = self._mmr_rerank(rec_ids, rec_scores, n=n)
        else:
            final_ids, final_scores = rec_ids[:n], rec_scores[:n]

        print(f'  -> {n_cand} candidates -> {len(final_ids)} recs '
              f'(mmr={use_mmr}, artist_limit={artist_limit})')
        df = pd.DataFrame([self._to_row(self.idx2item[i], s)
                           for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df) + 1)
        return df

    # ── 2. COLD-START ITEM ────────────────────────────────────────────────────

    def recommend_similar_to_new_item(self, track_name: str, artist_name: str,
                                       n: int = 10) -> pd.DataFrame:
        """
        For a NEW track not in catalog:
          1. TF-IDF content search -> top-K similar items in catalog
          2. Use their LightGCN embeddings as proxy vector
          3. FAISS search -> filter query -> return results
        """
        query = f'{track_name} {artist_name}'
        print(f'\n[new_item] "{track_name}" -- {artist_name}')

        cvec = self._text_to_content_vec(query)
        n_search = min(30, len(self._content_iids))
        _, cpos  = self._content_index.search(cvec, n_search)
        cand_iids = self._content_iids[cpos[0]]

        if len(cand_iids) == 0:
            print('  No similar items found in catalog')
            return pd.DataFrame()

        proxy_vec = self._proxy_vector_from_items(cand_iids[:15])
        scores, indices = self.index.search(proxy_vec, n * 3)
        query_lower = query.lower()

        final_ids, final_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            msid = self.idx2item[iid]
            meta = self.item_meta.get(msid, {})
            name = (meta.get('track_name', '') + ' ' +
                    meta.get('artist_name', '')).strip().lower()
            if name == query_lower:
                continue
            final_ids.append(int(iid)); final_scores.append(float(sc))
            if len(final_ids) >= n:
                break

        print(f'  -> {len(final_ids)} similar items found')
        df = pd.DataFrame([self._to_row(self.idx2item[i], s)
                           for i, s in zip(final_ids, final_scores)])
        df.insert(0, 'similar_to', f'{track_name} ({artist_name})')
        df.index = range(1, len(df) + 1)
        return df

    # ── 3. SIMILAR USERS (user-user CF) ──────────────────────────────────────

    def recommend_similar_users(self, user_id_str, n: int = 10,
                                 k_users: int = 20,
                                 filter_listened: bool = True) -> pd.DataFrame:
        """
        User-user CF:
          1. Find k nearest users in embedding space
          2. Aggregate their items (exclude listened)
          3. score = sum(sim_score) -> MMR rerank
        """
        print(f'\n[similar_users] User={user_id_str!r} | k={k_users}')
        tier, uid = self._get_user_tier(user_id_str)
        if uid is None:
            print('  User not in model -> popular items')
            return self.popular_items(n)

        self._ensure_user_index()
        sim_scores, nbs = self._user_index.search(
            self.user_vectors[uid:uid+1], k_users + 1)

        listened  = self._listened_set(user_id_str) if filter_listened else set()
        item_score: dict = defaultdict(float)
        for sim_uid, sim_sc in zip(nbs[0], sim_scores[0]):
            if int(sim_uid) == uid:
                continue
            for iid in self.train_matrix[int(sim_uid)].indices:
                if iid not in listened:
                    item_score[int(iid)] += float(sim_sc)

        if not item_score:
            print('  No recs from similar users -> fallback recommend_hybrid()')
            return self.recommend_hybrid(user_id_str, n=n)

        sorted_items = sorted(item_score.items(), key=lambda x: x[1], reverse=True)
        cand_ids = [i for i, _ in sorted_items[:n * 4]]
        cand_sc  = [s for _, s in sorted_items[:n * 4]]
        final_ids, final_scores = self._mmr_rerank(cand_ids, cand_sc, n=n)

        print(f'  {k_users} similar users -> {len(item_score)} cands -> top {len(final_ids)}')
        df = pd.DataFrame([self._to_row(self.idx2item[i], s)
                           for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df) + 1)
        return df

    # ── 4. TRENDING ───────────────────────────────────────────────────────────

    def recommend_trending(self, user_id_str=None, n: int = 10,
                            personal_weight: float = 0.5) -> pd.DataFrame:
        """
        Blend trending score with personal preference.
        personal_weight=0 -> pure trending | =1 -> pure personalized.
        user_id_str=None -> global trending only.
        """
        print(f'\n[trending] User={user_id_str!r} | personal_weight={personal_weight}')
        listened = self._listened_set(user_id_str) if user_id_str else set()

        if user_id_str is None or personal_weight == 0.0:
            trend_ids = np.argsort(self._trending_scores)[::-1]
            top_ids   = [int(i) for i in trend_ids if i not in listened][:n]
            df = pd.DataFrame([
                {**self._to_row(self.idx2item[i], self._trending_scores[i]),
                 'trending_rank': r + 1}
                for r, i in enumerate(top_ids)
            ])
            df.index = range(1, len(df) + 1)
            return df

        tier, uid = self._get_user_tier(user_id_str)
        query_vec = self.user_vectors[uid:uid+1] if uid is not None else None

        if query_vec is not None:
            scores, indices = self.index.search(query_vec, n * 8)
            blend: dict = {}
            for iid, sc in zip(indices[0], scores[0]):
                if iid in listened:
                    continue
                blend[int(iid)] = (personal_weight * float(sc)
                                   + (1.0 - personal_weight)
                                   * float(self._trending_scores[iid]))
        else:
            trend_ids = np.argsort(self._trending_scores)[::-1]
            blend = {int(i): float(self._trending_scores[i])
                     for i in trend_ids[:n * 5] if i not in listened}

        sorted_blend = sorted(blend.items(), key=lambda x: x[1], reverse=True)
        cand_ids = [i for i, _ in sorted_blend[:n * 3]]
        cand_sc  = [s for _, s in sorted_blend[:n * 3]]
        final_ids, final_scores = self._mmr_rerank(cand_ids, cand_sc, n=n)

        print(f'  -> {len(blend)} candidates -> top {len(final_ids)}')
        df = pd.DataFrame([self._to_row(self.idx2item[i], s)
                           for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df) + 1)
        return df

    # ── 5. DISCOVERY / SERENDIPITY ────────────────────────────────────────────

    def recommend_discovery(self, user_id_str, n: int = 10,
                             serendipity: float = 0.3) -> pd.DataFrame:
        """
        Explore outside comfort zone:
          - Find 'far users' (rank 50-100 in user space, not top-5)
          - Blend user_vec lightly with far_vec to shift the query
          - Low MMR lambda -> more diverse output
        serendipity: 0 = normal, 1 = fully random
        """
        print(f'\n[discovery] User={user_id_str!r} | serendipity={serendipity}')
        tier, uid = self._get_user_tier(user_id_str)
        if uid is None:
            return self.popular_items(n)

        self._ensure_user_index()
        n_search = min(150, self.user_vectors.shape[0])
        _, far_nbs = self._user_index.search(self.user_vectors[uid:uid+1], n_search)
        far_nbs    = far_nbs[0]

        base_vec = self.user_vectors[uid].copy()
        skip     = min(50, len(far_nbs) // 2)
        far_pool = far_nbs[skip:]

        if len(far_pool) > 0:
            far_vec = self.user_vectors[far_pool].mean(axis=0)
        else:
            rng     = np.random.default_rng(uid)
            far_vec = rng.standard_normal(self.dim).astype(np.float32)
            far_vec /= (np.linalg.norm(far_vec) + 1e-8)

        fused     = (1.0 - serendipity) * base_vec + serendipity * far_vec
        query_vec = self._normalize(fused)

        scores, indices = self.index.search(query_vec, n * 6)
        listened = self._listened_set(user_id_str)
        rec_ids, rec_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            if iid not in listened:
                rec_ids.append(int(iid)); rec_scores.append(float(sc))

        final_ids, final_scores = self._mmr_rerank(rec_ids, rec_scores, n=n, lambda_=0.4)
        print(f'  {len(rec_ids)} candidates -> {len(final_ids)} discovery tracks')
        df = pd.DataFrame([self._to_row(self.idx2item[i], s)
                           for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df) + 1)
        return df

    # ── 6. SESSION-BASED (next-track prediction) ──────────────────────────────

    def recommend_next_in_session(self, session_msids: list, n: int = 10,
                                   decay: float = 0.8,
                                   filter_session: bool = True) -> pd.DataFrame:
        """
        Predict next track from current session.
        session_msids: list[msid] in chronological order (old -> new).
        decay: recent tracks get higher weight (0.8 -> each step x0.8).
        """
        print(f'\n[session] {len(session_msids)} tracks in session')
        found = [(m, self.item2idx[m]) for m in session_msids if m in self.item2idx]
        if not found:
            print('  No tracks found in catalog -> popular items')
            return self.popular_items(n)

        n_f   = len(found)
        w     = np.array([decay ** (n_f - 1 - k) for k in range(n_f)], dtype=np.float32)
        w    /= w.sum()
        iids  = np.array([iid for _, iid in found], dtype=np.int32)

        query_vec = self._proxy_vector_from_items(iids, weights=w)
        scores, indices = self.index.search(query_vec, n * 4)

        session_set = {iid for _, iid in found} if filter_session else set()
        rec_ids, rec_scores = [], []
        for iid, sc in zip(indices[0], scores[0]):
            if iid not in session_set:
                rec_ids.append(int(iid)); rec_scores.append(float(sc))

        final_ids, final_scores = self._mmr_rerank(rec_ids, rec_scores, n=n, lambda_=0.7)
        print(f'  {n_f}/{len(session_msids)} tracks recognized -> '
              f'{len(final_ids)} next tracks')
        df = pd.DataFrame([self._to_row(self.idx2item[i], s)
                           for i, s in zip(final_ids, final_scores)])
        df.index = range(1, len(df) + 1)
        return df

    # ── 7. BY ARTIST (+ expand similar artists) ───────────────────────────────

    def recommend_by_artist(self, artist_name: str, n: int = 10,
                             expand: bool = True,
                             k_similar_artists: int = 5) -> pd.DataFrame:
        """
        Recommend tracks from artist + (if expand) similar artists.
        """
        print(f'\n[by_artist] "{artist_name}" | expand={expand}')
        a_lower  = artist_name.strip().lower()
        all_iids = list(self._artist2items.get(a_lower, []))
        print(f'  Direct items: {len(all_iids)}')

        if expand:
            a_pos = self._artist_name_to_pos.get(a_lower)
            if a_pos is not None:
                a_vec = self._artist_vecs[a_pos:a_pos+1]
            elif all_iids:
                a_vec = self._proxy_vector_from_items(all_iids[:30])
            else:
                cvec = self._text_to_content_vec(a_lower)
                _, cpos = self._content_index.search(cvec, 10)
                cand = self._content_iids[cpos[0]].tolist()
                a_vec = self._proxy_vector_from_items(cand) if cand else None

            if a_vec is not None:
                vec_2d = a_vec if a_vec.ndim == 2 else a_vec.reshape(1, -1)
                sim_sc, sim_pos = self._artist_index.search(
                    vec_2d, k_similar_artists + 1)
                for pos, sc in zip(sim_pos[0], sim_sc[0]):
                    sim_name = self._artist_names[int(pos)]
                    if sim_name == a_lower:
                        continue
                    sim_iids = self._artist2items.get(sim_name, [])
                    all_iids.extend(sim_iids[:8])
                    print(f'    + Similar: "{sim_name}" ({sc:.3f}) '
                          f'-- {len(sim_iids)} tracks')

        if not all_iids:
            print('  No artist or tracks found in catalog')
            return pd.DataFrame()

        item_counts = np.asarray(self.train_matrix.sum(axis=0)).flatten()
        unique_iids = list(dict.fromkeys(all_iids))
        sorted_iids = sorted(unique_iids, key=lambda i: item_counts[i], reverse=True)
        top_iids    = sorted_iids[:n]
        top_scores  = [float(item_counts[i]) for i in top_iids]

        df = pd.DataFrame([self._to_row(self.idx2item[i], s)
                           for i, s in zip(top_iids, top_scores)])
        df['artist_name'] = [self._iid_to_artist.get(int(i), '') for i in top_iids]
        df.index = range(1, len(df) + 1)
        return df


# ── Instantiate ───────────────────────────────────────────────────────────────
print('Initializing HybridRecommender...')
hybrid_rec = HybridRecommender(CONFIG['output_dir'], cold_threshold=COLD_THRESHOLD)


## 9. Demo Hybrid + Enhanced Inference


In [ ]:
from collections import Counter

# ── Pick demo user ─────────────────────────────────────────────────────────────
test_uid_idx = np.unique(hybrid_rec.test_matrix.nonzero()[0])[0]
test_user    = hybrid_rec.idx2user[test_uid_idx]
tier, _      = hybrid_rec._get_user_tier(test_user)
uid_ts       = hybrid_rec.user_item_ts_matrix.get(test_uid_idx, {})

print('=' * 62)
print(f'  Demo User : {test_user}')
print(f'  Tier      : {tier}')
print(f'  Train     : {hybrid_rec.train_matrix[test_uid_idx].nnz} tracks')
print(f'  Test      : {hybrid_rec.test_matrix[test_uid_idx].nnz} tracks')
if uid_ts:
    ts_vals = list(uid_ts.values())
    print(f'  Timestamp : {pd.Timestamp(min(ts_vals), unit="s").date()} -> '
          f'{pd.Timestamp(max(ts_vals), unit="s").date()}')
print('=' * 62)

# ── 1. Hybrid Recommend (warm user) ───────────────────────────────────────────
print('\n[1] Hybrid Recommend (warm) -- LightGCN + artist diversity + trending')
display(hybrid_rec.recommend_hybrid(
    test_user, n=10, artist_limit=2, trending_boost=0.1
))

# ── 2. Cold-Start User (completely new) ───────────────────────────────────────
print('\n[2] Cold-Start User -- only liked track/artist hints provided')
display(hybrid_rec.recommend_hybrid(
    user_id_str   = 'NEW_USER_001',
    n             = 10,
    liked_tracks  = ['Lightbulb Sun', 'Loner', 'Black Hole Sun'],
    liked_artists = ['Radiohead'],
))

# ── 3. Lukewarm User (few interactions) ───────────────────────────────────────
lukewarm_user = None
for u_str in hybrid_rec.idx2user[:500]:
    _tier, _uid = hybrid_rec._get_user_tier(u_str)
    if _tier == 'lukewarm':
        lukewarm_user = u_str
        break

if lukewarm_user:
    print(f'\n[3] Lukewarm User "{lukewarm_user}" -- blend LightGCN + item-proxy')
    display(hybrid_rec.recommend_hybrid(lukewarm_user, n=10))
else:
    print(f'\n[3] No lukewarm user in first 500 '
          f'(try lowering COLD_THRESHOLD from {COLD_THRESHOLD})')

# ── 4. Cold-Start Item (new song not in catalog) ──────────────────────────────
print('\n[4] Cold-Start Item -- recommend similar to an unseen track')
display(hybrid_rec.recommend_similar_to_new_item(
    track_name  = 'Creep',
    artist_name = 'Radiohead',
    n           = 10
))

# ── 5. Similar Users (user-user CF) ───────────────────────────────────────────
print('\n[5] Similar Users (user-user Collaborative Filtering)')
display(hybrid_rec.recommend_similar_users(test_user, n=10, k_users=25))

# ── 6. Trending ───────────────────────────────────────────────────────────────
print('\n[6a] Global Trending (no personalization)')
display(hybrid_rec.recommend_trending(n=10))

print('\n[6b] Personalized Trending (50% personal + 50% trending)')
display(hybrid_rec.recommend_trending(test_user, n=10, personal_weight=0.5))

# ── 7. Discovery / Serendipity ────────────────────────────────────────────────
print('\n[7] Discovery -- explore outside comfort zone (serendipity=0.35)')
display(hybrid_rec.recommend_discovery(test_user, n=10, serendipity=0.35))

# ── 8. Session-based ──────────────────────────────────────────────────────────
if uid_ts:
    sorted_by_ts  = sorted(uid_ts.items(), key=lambda x: x[1], reverse=True)
    session_iids  = [iid for iid, _ in sorted_by_ts[:5]]
    session_msids = [hybrid_rec.idx2item[i] for i in session_iids
                     if i < len(hybrid_rec.idx2item)]
    print(f'\n[8] Session-based -- next track after {len(session_msids)} recent tracks')
    display(hybrid_rec.recommend_next_in_session(session_msids, n=10, decay=0.8))
else:
    print('\n[8] Skip session-based demo (no timestamp data available)')

# ── 9. By Artist (+ similar artist expand) ─────────────────────────────────────
artist_pop = Counter(
    hybrid_rec._iid_to_artist.get(iid, '')
    for iid in range(hybrid_rec.item_vectors.shape[0])
)
artist_pop.pop('', None)
top_artist = artist_pop.most_common(1)[0][0] if artist_pop else 'radiohead'

print(f'\n[9] By Artist: "{top_artist}" (+ expand to similar artists)')
display(hybrid_rec.recommend_by_artist(top_artist, n=10, expand=True))

print('\n--- Demo complete! ---')
